# 4.3 Data Cleaning Techniques

**Machine Learning Mastery: Mathematics to MLOps** · Chapter 4: Data Preprocessing & Exploratory Data Analysis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RithikRaj129/ml-mastery/blob/main/chapter-04-data-preprocessing-and-exploratory-data-analysis/04.03_data-cleaning-techniques.ipynb)

Companion code for Section 4.3 of the book. Run the cells in order from top to bottom.

**Example 4.3.1**

In [ ]:
import pandas as pd
import numpy as np

# Simulate a messy dataset with duplicates
df = pd.DataFrame({
    'customer_id': [1,2,3,2,4,5,3,6],
    'age':         [25,32,28,32,45,28,28,31],
    'income':      [45000,72000,58000,72000,95000,58000,58000,61000],
    'churned':     [0,0,1,0,0,1,1,0]
})

print('Original shape:', df.shape)
print('Duplicate rows:', df.duplicated().sum())
print('Duplicate customer_ids:', df.duplicated(subset=['customer_id']).sum())

# View duplicates
print('\nDuplicate rows:')
print(df[df.duplicated(keep=False)].sort_values('customer_id'))

# Remove exact duplicates (all columns identical)
df_clean = df.drop_duplicates()
print(f'\nAfter removing exact duplicates: {df_clean.shape}')

# Remove duplicates by key column (keep first occurrence)
df_dedup = df.drop_duplicates(subset=['customer_id'], keep='first')
print(f'After deduplication by customer_id: {df_dedup.shape}')

# Fuzzy duplicate detection - find similar (not exact) records
# e.g., 'John Smith' and 'Jon Smith' - same person?
names = pd.Series(['John Smith', 'Jon Smith', 'Jane Doe', 'Jane Doe', 'J Smith'])
# Flag records that are very similar (edit distance < 3)
from difflib import get_close_matches
for name in names:
    matches = get_close_matches(name, names, n=3, cutoff=0.8)
    if len(matches) > 1:
        print(f'"{name}" is similar to: {[m for m in matches if m != name]}')

**Example 4.3.2**

In [ ]:
import pandas as pd
import numpy as np

# Common real-world data type and format problems
df = pd.DataFrame({
    'age':      ['25', '32', 'thirty', '28', None, '45.5'],  # Should be numeric
    'income':   ['$45,000', '$72,000', '58000', '$95,000', '0', ''],
    'date':     ['2023-01-15', '15/01/2023', 'Jan 15 2023', '2023/01/15', None, 'INVALID'],
    'churned':  ['Yes', 'No', 'YES', '1', '0', 'no'],  # Inconsistent booleans
    'gender':   ['Male', 'female', 'M', 'F', 'MALE', 'Female'],  # Inconsistent strings
})

print('Original dtypes:'); print(df.dtypes)

# Fix: parse age (convert to numeric, coerce errors to NaN)
df['age_clean'] = pd.to_numeric(df['age'], errors='coerce')
print(f'\nAge parsing errors (→NaN): {df.age_clean.isnull().sum()}')

# Fix: clean income (remove $, commas, convert to float)
df['income_clean'] = (
    df['income']
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False)
    .str.strip()
    .replace('', np.nan)
    .pipe(pd.to_numeric, errors='coerce')
)

# Fix: parse dates flexibly
df['date_clean'] = pd.to_datetime(df['date'], errors='coerce', infer_datetime_format=True)
print(f'Date parsing errors: {df.date_clean.isnull().sum()}')

# Fix: standardise boolean column
bool_map = {'yes': 1, 'no': 0, '1': 1, '0': 0}
df['churned_clean'] = df['churned'].str.lower().map(bool_map)

# Fix: standardise categorical strings
gender_map = {'male': 'Male', 'm': 'Male', 'female': 'Female', 'f': 'Female'}
df['gender_clean'] = df['gender'].str.lower().map(gender_map)

print('\nCleaned columns:')
print(df[['age_clean','income_clean','date_clean','churned_clean','gender_clean']])

**Example 4.3.3**

In [ ]:
import pandas as pd
import numpy as np

def validate_dataset(df):
    '''Validate a dataset against domain rules. Returns a validation report.'''
    errors = []

    # Rule 1: Age must be in [0, 120]
    if 'age' in df.columns:
        invalid_age = df[(df['age'] < 0) | (df['age'] > 120)]
        if len(invalid_age):
            errors.append(f'Age out of range [0,120]: {len(invalid_age)} rows')

    # Rule 2: Income must be non-negative
    if 'income' in df.columns:
        neg_income = df[df['income'] < 0]
        if len(neg_income):
            errors.append(f'Negative income: {len(neg_income)} rows')

    # Rule 3: Target must be binary {0, 1}
    if 'churned' in df.columns:
        invalid_target = df[~df['churned'].isin([0, 1])]
        if len(invalid_target):
            errors.append(f'Invalid target values: {len(invalid_target)} rows')

    # Rule 4: No future dates in historical data
    if 'date' in df.columns:
        future_dates = df[df['date'] > pd.Timestamp.now()]
        if len(future_dates):
            errors.append(f'Future dates in historical data: {len(future_dates)} rows')

    # Report
    if errors:
        print('VALIDATION FAILED:')
        for e in errors: print(f'    {e}')
    else:
        print('✓ All validation checks passed.')
    return errors

# Test with a messy dataset
test_df = pd.DataFrame({
    'age':     [25, -1, 32, 150, 28],
    'income':  [45000, 72000, -500, 95000, 58000],
    'churned': [0, 1, 0, 2, 1],   # '2' is invalid
})

errors = validate_dataset(test_df)